# ECE 285 - HW1: Chest X-Ray VAE 

In [ ]:
!pip install -q pytorch-ignite kagglehub

import os
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))

In [2]:
import os
import kagglehub

# Default local path or download directly via kagglehub
DATA_ROOT = "./data/chest_xray"

if not os.path.exists(os.path.join(DATA_ROOT, "train")):
    print("Downloading chest_xray dataset from Kaggle via kagglehub...")
    downloaded_path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
    for root, dirs, _ in os.walk(downloaded_path):
        if {"train", "val", "test"}.issubset(set(dirs)):
            DATA_ROOT = root
            break

print(f"Ready! Using DATA_ROOT = {DATA_ROOT}")
for split in ["train", "val", "test"]:
    split_path = os.path.join(DATA_ROOT, split)
    print(f"  - {split}: exists={os.path.exists(split_path)} ({split_path})")

## 2. Dataset & DataLoader

In [3]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import PIL.Image as Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import utils

from ignite.engine import Engine
from ignite.metrics import FID, InceptionScore


class ChestXRayDataset(Dataset):
    """
    Loads all images from `root_dir/<split>/*/*` (ignoring class folders).
    """
    def __init__(self, root_dir, split="train", img_size=128):
        self.img_size = img_size
        self.image_paths = []
        for ext in ("*.jpeg", "*.jpg", "*.png"):
            pattern = os.path.join(root_dir, split, "*", ext)
            self.image_paths.extend(glob.glob(pattern))
        self.image_paths.sort()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("L")  # PIL Image (W, H), 1-channel grayscale
        img = img.resize((self.img_size, self.img_size))      # PIL Image (img_size, img_size)
        x = torch.tensor(np.array(img), dtype=torch.float32) / 255.0  # (img_size, img_size), values in [0, 1]
        x = x.unsqueeze(0)                                    # (1, img_size, img_size)
        return x


def get_dataloaders(root_dir, img_size=128, batch_size=64):
    train_ds = ChestXRayDataset(root_dir, split="train", img_size=img_size)
    val_ds   = ChestXRayDataset(root_dir, split="val",   img_size=img_size)
    test_ds  = ChestXRayDataset(root_dir, split="test",  img_size=img_size)

    print(f"Dataset counts -> Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader

## 3. VAE Architecture & Loss Function

In [4]:
class VAE(nn.Module):
    def __init__(self, latent_dim=64, device=DEVICE):
        super().__init__()
        self.latent_dim = latent_dim
        self.device = device

        # Encoder: (B, 1, 128, 128) -> (B, 64, 16, 16)
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),    # (B, 16, 64, 64)
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2),

            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),   # (B, 32, 32, 32)
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),   # (B, 64, 16, 16)
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
        )

        self.fc_mu = nn.Linear(64 * 16 * 16, latent_dim)             # (B, 16384) -> (B, latent_dim)
        self.fc_logvar = nn.Linear(64 * 16 * 16, latent_dim)         # (B, 16384) -> (B, latent_dim)

        # Decoder: (B, latent_dim) -> (B, 64, 16, 16) -> (B, 1, 128, 128)
        self.fc_decode = nn.Linear(latent_dim, 64 * 16 * 16)         # (B, latent_dim) -> (B, 16384)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # (B, 32, 32, 32)
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),  # (B, 16, 64, 64)
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),   # (B, 1, 128, 128)
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)               # (B, 64, 16, 16)
        h = torch.flatten(h, start_dim=1) # (B, 16384)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        h = F.relu(self.fc_decode(z))
        h = h.view(-1, 64, 16, 16)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar


def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    batch_size = x.size(0)
    num_elements = x.numel()  # batch_size * C * H * W

    # Sum over pixels/latent dims (per sample) for backpropagation
    recon_sum = F.mse_loss(recon_x, x, reduction="sum") / batch_size
    kl_sum = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size
    loss_sum = recon_sum + beta * kl_sum

    # Mean over all elements for printing and plotting
    loss_mean = (loss_sum * batch_size) / num_elements
    return loss_sum, loss_mean, recon_sum, kl_sum

## 4. Training, Plotting & Evaluation Functions

In [5]:
def plot_loss_curves(train_losses, val_losses, start_epoch=0, save_path="loss_curves.png", show=False):
    plt.figure(figsize=(7, 4))
    epochs_range = range(start_epoch + 1, start_epoch + len(train_losses) + 1)
    plt.plot(epochs_range, train_losses, label="Train Loss (Mean)")
    plt.plot(epochs_range, val_losses, label="Val Loss (Mean)")
    plt.xlabel("Epoch")
    plt.ylabel("Mean VAE Loss (MSE + KL)")
    plt.title("Training and Validation Loss Curves")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    if show:
        plt.show()
    plt.close()


def train_vae(
    model,
    train_loader,
    val_loader,
    test_loader=None,
    start_epoch=0,
    epochs=50,
    lr=2e-4,
    min_lr=1e-6,
    beta=1.0,
    eval_every_k=10,
    device=DEVICE,
    save_dir="./",
    resume_checkpoint=None,
):
    model.to(device)

    # Load weights from checkpoint if specified
    if resume_checkpoint and os.path.exists(resume_checkpoint):
        ckpt = torch.load(resume_checkpoint, map_location=device)
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            model.load_state_dict(ckpt["model_state_dict"])
        elif isinstance(ckpt, dict):
            model.load_state_dict(ckpt)
        print(f"Successfully loaded checkpoint: {resume_checkpoint} (Resuming at epoch {start_epoch + 1})")
    elif resume_checkpoint:
        print(f"Warning: Checkpoint '{resume_checkpoint}' not found! Starting without loading.")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = max(1, epochs - start_epoch)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=min_lr)

    train_losses, val_losses = [], []
    loss_plot_path = os.path.join(save_dir, "loss_curves.png")
    eval_log_path = os.path.join(save_dir, "test_eval_metrics.txt")

    for epoch in range(start_epoch + 1, epochs + 1):
        # --- Train ---
        model.train()
        running_train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch [{epoch}/{epochs}]", leave=False)
        for x in pbar:
            x = x.to(device)
            optimizer.zero_grad()

            recon_x, mu, logvar = model(x)
            loss_sum, loss_mean, _, _ = vae_loss(recon_x, x, mu, logvar, beta=beta)

            # Backpropagate using sum loss
            loss_sum.backward()
            optimizer.step()

            # Track mean loss for printing & plotting
            running_train_loss += loss_mean.item()
            pbar.set_postfix(loss=f"{loss_mean.item():.5f}")

        avg_train_loss = running_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # --- Validate ---
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for x in val_loader:
                x = x.to(device)
                recon_x, mu, logvar = model(x)
                _, loss_mean, _, _ = vae_loss(recon_x, x, mu, logvar, beta=beta)
                running_val_loss += loss_mean.item()

        avg_val_loss = running_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        print(f"Epoch [{epoch}/{epochs}] | LR: {current_lr:.2e} | Train Loss (Mean): {avg_train_loss:.5f} | Val Loss (Mean): {avg_val_loss:.5f}")
        plot_loss_curves(train_losses, val_losses, start_epoch=start_epoch, save_path=loss_plot_path, show=(epoch == epochs))

        # --- Every k epochs: save weights, generate 5 images, evaluate & log test metrics ---
        if eval_every_k and (epoch % eval_every_k == 0 or epoch == epochs):
   
            ckpt_path = os.path.join(save_dir, f"vae_chest_xray_epoch_{epoch}.pt")
            latest_path = os.path.join(save_dir, "vae_chest_xray.pt")
            torch.save(model.state_dict(), ckpt_path)
            torch.save(model.state_dict(), latest_path)
            print(f"  -> Saved model checkpoint to {ckpt_path}")

            samples_path = os.path.join(save_dir, f"generated_samples_epoch_{epoch}.png")
            generate_images(model, num_images=5, device=device, save_path=samples_path)

            if test_loader is not None:
                model.eval()
                running_test_loss = 0.0
                with torch.no_grad():
                    for x in test_loader:
                        x = x.to(device)
                        recon_x, mu, logvar = model(x)
                        _, loss_mean, _, _ = vae_loss(recon_x, x, mu, logvar, beta=beta)
                        running_test_loss += loss_mean.item()
                avg_test_loss = running_test_loss / len(test_loader)
                print(f"  -> [Test Eval @ Epoch {epoch}] Test Loss (Mean): {avg_test_loss:.5f}")
                incep_score, fid = compute_fid_and_is(model, test_loader, device=device)

                log_line = (
                    f"Epoch {epoch:03d}/{epochs:03d} | LR: {current_lr:.2e} | "
                    f"Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | "
                    f"Test Loss: {avg_test_loss:.6f} | IS: {incep_score:.4f} | FID: {fid:.4f}\n"
                )
                with open(eval_log_path, "a") as f:
                    f.write(log_line)
                print(f"  -> Logged test metrics to {eval_log_path}")

    return train_losses, val_losses


def interpolate(batch):
    """Resize grayscale (B, 1, H, W) to 3-channel (B, 3, 299, 299) for InceptionV3."""
    if batch.size(1) == 1:
        batch = batch.repeat(1, 3, 1, 1)
    out = F.interpolate(batch, size=(299, 299), mode="bicubic", align_corners=False)
    return out.clamp(0.0, 1.0)


def compute_fid_and_is(model, test_loader, device=DEVICE):
    model.to(device)
    model.eval()

    def evaluation_step(engine, batch):
        with torch.no_grad():
            z = torch.randn(batch.size(0), model.latent_dim, device=device)
            generated = model.decode(z).cpu()
            fake = interpolate(generated)
            real = interpolate(batch)
            return fake, real

    fid_metric = FID(device=device)
    is_metric = InceptionScore(device=device, output_transform=lambda x: x[0])

    evaluator = Engine(evaluation_step)
    fid_metric.attach(evaluator, "fid")
    is_metric.attach(evaluator, "is")

    evaluator.run(test_loader)
    metrics = evaluator.state.metrics
    print(f"  -> Inception Score (IS): {metrics['is']:.4f} | FID: {metrics['fid']:.4f}")
    return metrics["is"], metrics["fid"]


def generate_images(model, num_images=5, device=DEVICE, save_path="generated_samples.png"):
    model.to(device)
    model.eval()
    with torch.no_grad():
        z = torch.randn(num_images, model.latent_dim, device=device)
        samples = model.decode(z).cpu()

    nrow = min(num_images, 5)
    grid = utils.make_grid(samples, nrow=nrow, padding=2)
    plt.figure(figsize=(2.2 * nrow, 2.5))
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.axis("off")
    plt.title(f"Generated Chest X-Ray Images ({num_images} samples)")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()
    plt.close()

## 5. Run VAE Training & Evaluation

In [ ]:
HYPERPARAMS = {
    "root_dir": DATA_ROOT,
    "img_size": 128,
    "batch_size": 32,
    "latent_dim": 128,
    "lr": 4e-4,
    "min_lr": 4e-5,
    "start_epoch": 0,
    "epochs": 400,
    "beta": 1.0,
    "eval_every_k": 25,
}
print("Hyperparameters:", HYPERPARAMS)
print(f"Using device: {DEVICE}")

# Load data
train_loader, val_loader, test_loader = get_dataloaders(
    root_dir=HYPERPARAMS["root_dir"],
    img_size=HYPERPARAMS["img_size"],
    batch_size=HYPERPARAMS["batch_size"],
)

# Initialize & train VAE
model = VAE(latent_dim=HYPERPARAMS["latent_dim"], device=DEVICE).to(DEVICE)
train_losses, val_losses = train_vae(
    model,
    train_loader,
    val_loader,
    test_loader=test_loader,
    start_epoch=HYPERPARAMS.get("start_epoch", 0),
    epochs=HYPERPARAMS["epochs"],
    lr=HYPERPARAMS["lr"],
    min_lr=HYPERPARAMS["min_lr"],
    beta=HYPERPARAMS["beta"],
    eval_every_k=HYPERPARAMS["eval_every_k"],
    resume_checkpoint=HYPERPARAMS.get("resume_checkpoint", None),
    device=DEVICE,
    save_dir="./",
)

# Save final model weights & outputs
weights_path = "vae_chest_xray.pt"
samples_path = "generated_samples.png"
torch.save(model.state_dict(), weights_path)
print(f"Saved model weights to {weights_path}")

# Generate sample chest X-rays
generate_images(model, num_images=5, device=DEVICE, save_path=samples_path)

# Evaluate IS and FID on test set
compute_fid_and_is(model, test_loader, device=DEVICE)